# Emergency Vehicle Classification — Optimized Pipeline
## YOLOv8s-cls Fine-Tuning on Google Colab GPU

This notebook trains a **YOLOv8s-cls** emergency vehicle classifier on Colab.
Modelled after the [trafficDetection](./trafficDetection.ipynb) notebook for fast, reliable Colab execution.

**Pipeline:**
1. **Colab Setup** — Install ultralytics
2. **Download & Prepare Dataset** — Clone from GitHub, sort via CSV, split train/val
3. **Imports & Config** — Lean imports, paths on local `/content/` for fast I/O
4. **Train** — YOLOv8s-cls with `cache=True`, `freeze=9`, AdamW, cosine LR
5. **Evaluate** — `model.val()` + sample inference
6. **Save to Google Drive** — Persist weights to Drive at the end

### Colab Setup

In [1]:
%pip -q install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.0 MB/s eta 0:00:0000:01


### Download & Prepare Dataset
- Clone the [Emergency Vehicles on Road Networks](https://github.com/Shatnawi-Moath/EMERGENCY-VEHICLES-ON-ROAD-NETWORKS-A-NOVEL-GENERATED-DATASET-USING-GANs) dataset
- Read `train1.csv` (comma-separated) to identify emergency (1) vs non-emergency (0) labels
- Sort images into `train/emergency/`, `train/non_emergency/`, `val/emergency/`, `val/non_emergency/`
- 80/20 train/val split

In [2]:
import csv
import os
import random
import shutil
from pathlib import Path

# ── 1. Clone dataset repo (shallow) ──
REPO_URL = "https://github.com/Shatnawi-Moath/EMERGENCY-VEHICLES-ON-ROAD-NETWORKS-A-NOVEL-GENERATED-DATASET-USING-GANs.git"
RAW_DIR = Path("/content/data/raw/emergency-vehicles")

if not RAW_DIR.exists():
    os.system(f"git clone --depth 1 {REPO_URL} {RAW_DIR}")
    print(f"✓ Cloned to {RAW_DIR}")
else:
    print(f"✓ Already cloned at {RAW_DIR}")

# ── 2. Resolve paths robustly (handles case/space differences) ──
# CSV is in: Src/Dataset/train1.csv (based on repo structure)
CSV_PATH = RAW_DIR / "Src" / "Dataset" / "train1.csv"

# Images are in: Src/Dataset/Main Dataset/
# (some people mistakenly use DataSet/Main_Dataset; this fixes that)
CANDIDATE_IMAGE_DIRS = [
    RAW_DIR / "Src" / "Dataset" / "Main Dataset",
    RAW_DIR / "Src" / "Dataset" / "Main_Dataset",
    RAW_DIR / "Src" / "DataSet" / "Main Dataset",
    RAW_DIR / "Src" / "DataSet" / "Main_Dataset",
]

IMAGE_DIR = next((p for p in CANDIDATE_IMAGE_DIRS if p.exists()), None)
if IMAGE_DIR is None:
    raise FileNotFoundError(
        "Could not find image directory. Tried:\n" + "\n".join(str(p) for p in CANDIDATE_IMAGE_DIRS)
    )

if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found at: {CSV_PATH}")

print(f"✓ Using CSV: {CSV_PATH}")
print(f"✓ Using IMAGE_DIR: {IMAGE_DIR}")

# ── 3. Read CSV labels ──
# CSV is comma-separated: "Image (1).jpeg,1"
labels = {}
with open(CSV_PATH, "r", newline="") as f:
    reader = csv.reader(f)  # standard comma delimiter
    header = next(reader, None)  # skip header
    for row in reader:
        if len(row) >= 2:
            img_name = row[0].strip()
            try:
                label = int(row[1].strip())
            except ValueError:
                continue  # skip malformed rows
            labels[img_name] = "emergency" if label == 1 else "non_emergency"

print(f"✓ Read {len(labels)} labels from CSV")
em_count = sum(1 for v in labels.values() if v == "emergency")
print(f"  emergency: {em_count}  |  non_emergency: {len(labels) - em_count}")

# ── 4. Sort into train/val with 80/20 split (cap at 1000 images total) ──
DATASET_DIR = Path("/content/data/dataset")
MAX_IMAGES = 1000  # total cap across both classes

for split in ["train", "val"]:
    for cls in ["emergency", "non_emergency"]:
        (DATASET_DIR / split / cls).mkdir(parents=True, exist_ok=True)

random.seed(42)
by_class = {"emergency": [], "non_emergency": []}

for img_name, cls in labels.items():
    src = IMAGE_DIR / img_name
    if src.exists():
        by_class[cls].append(src)
    else:
        matches = [p for p in IMAGE_DIR.glob("*") if p.name.lower() == img_name.lower()]
        if matches:
            by_class[cls].append(matches[0])

# Cap to MAX_IMAGES total, balanced across classes
per_class_cap = MAX_IMAGES // 2
for cls in by_class:
    random.shuffle(by_class[cls])
    by_class[cls] = by_class[cls][:per_class_cap]

total = sum(len(v) for v in by_class.values())
print(f"  Using {total} images (capped at {MAX_IMAGES})")

for cls in ["emergency", "non_emergency"]:
    imgs = by_class[cls]
    random.shuffle(imgs)
    split_idx = int(len(imgs) * 0.8)

    train_imgs = imgs[:split_idx]
    val_imgs = imgs[split_idx:]

    for img in train_imgs:
        shutil.copy2(img, DATASET_DIR / "train" / cls / img.name)
    for img in val_imgs:
        shutil.copy2(img, DATASET_DIR / "val" / cls / img.name)

    print(f"  {cls}: {len(train_imgs)} train / {len(val_imgs)} val")

# ── 5. Copy test dataset (for later inference) ──
# Folder in repo is "Test Dataset" (with space)
TEST_CANDIDATES = [
    RAW_DIR / "Src" / "Test Dataset",
    RAW_DIR / "Src" / "Test_Dataset",
]
TEST_SRC = next((p for p in TEST_CANDIDATES if p.exists()), TEST_CANDIDATES[0])
TEST_DIR = Path("/content/data/test_images")

if TEST_SRC.exists():
    shutil.copytree(TEST_SRC, TEST_DIR, dirs_exist_ok=True)
    test_count = len([p for p in TEST_DIR.rglob("*") if p.is_file()])
    print(f"  test images: {test_count} (copied to {TEST_DIR})")
else:
    print(f"  ⚠️ Test_Dataset not found at {TEST_SRC}")

# ── 6. Verify counts ──
for split in ["train", "val"]:
    for cls in ["emergency", "non_emergency"]:
        n = len(list((DATASET_DIR / split / cls).glob("*")))
        print(f"  {split}/{cls}: {n}")

print(f"\n✓ Dataset ready at {DATASET_DIR}")


✓ Cloned to /content/data/raw/emergency-vehicles
✓ Using CSV: /content/data/raw/emergency-vehicles/Src/Dataset/train1.csv
✓ Using IMAGE_DIR: /content/data/raw/emergency-vehicles/Src/Dataset/Main Dataset
✓ Read 20000 labels from CSV
  emergency: 10000  |  non_emergency: 10000
  Using 1000 images (capped at 1000)
  emergency: 400 train / 100 val
  non_emergency: 400 train / 100 val
  test images: 101 (copied to /content/data/test_images)
  train/emergency: 400
  train/non_emergency: 400
  val/emergency: 100
  val/non_emergency: 100

✓ Dataset ready at /content/data/dataset


In [12]:
!ls

data  sample_data


In [3]:
import os
import random
import shutil
from pathlib import Path

import cv2
import numpy as np
import torch
from ultralytics import YOLO

# Reproducibility
random.seed(42)
np.random.seed(42)

# Paths — all on local /content/ for fast I/O
DATASET_DIR = Path("/content/data/dataset")
TEST_DIR = Path("/content/data/test_images")
MODEL_SAVE = Path("/content/models")
MODEL_SAVE.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["emergency", "non_emergency"]

print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ No GPU — switch to GPU runtime: Runtime → Change runtime type → T4 GPU")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPU Available: True
GPU: Tesla T4
VRAM: 14.6 GB


### Train YOLOv8s-cls

Optimized config mirroring `trafficDetection.ipynb`:
- **YOLOv8s-cls** (small, fast, good accuracy with fine-tuning)
- **`cache=True`** — loads dataset into RAM (huge speedup)
- **`freeze=9`** — freeze backbone (layers 0-8), only train classification head (layer 9)
- **`workers=4`** — maximize CPU data loading
- **AdamW + cosine LR** — stable convergence
- **`patience=15`** — early stopping

In [4]:
# Model config
MODEL_NAME = "yolov8s-cls"
EPOCHS = 80
IMG_SIZE = 448
BATCH_SIZE = 16

# Load pretrained model (small — fast fine-tuning)
model = YOLO(f"{MODEL_NAME}.pt")

# Train with resource optimization (mirrors trafficDetection.ipynb pattern)
results = model.train(
    plots=True,
    data=str(DATASET_DIR),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,
    patience=15,  # early stopping
    lr0=0.001,  # slightly lower LR to avoid overfitting spikes
    cos_lr=True,  # smooth LR decay
    save=True,
    workers=4,
    cache=True,  # load dataset into RAM — massive speedup
    optimizer="AdamW",
    freeze=9,  # freeze backbone (layers 0-8) — keep Classify head (layer 9) trainable
    close_mosaic=10,
    # Augmentations tuned for traffic scenes (same as trafficDetection)
    hsv_h=0.01,
    hsv_s=0.6,
    hsv_v=0.4,
    degrees=3.0,  # vehicles rarely rotate
    translate=0.07,
    scale=0.4,
    shear=0.0,
    flipud=0.0,  # never flip traffic vertically
    fliplr=0.5,  # horizontal flip is realistic
    label_smoothing=0.1,  # prevents overconfident predictions
    weight_decay=0.0005,
    dropout=0.0,
)

print("\n✓ Training completed")
print(f"Results saved to: {results.save_dir if hasattr(results, 'save_dir') else 'runs/classify'}")

WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/data/dataset, degrees=3.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=9, half=False, hsv_h=0.01, hsv_s=0.6, hsv_v=0.4, imgsz=448, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=F

### Evaluation & Metrics

In [ ]:
# Evaluate on validation set using YOLO's built-in validation
val_results = model.val(data=str(DATASET_DIR), split="val", verbose=True)

print("\n" + "=" * 60)
print("EVALUATION RESULTS (val split)")
print("=" * 60)
print(f"  Top-1 Accuracy: {val_results.top1:.4f} ({val_results.top1 * 100:.2f}%)")
print(f"  Top-5 Accuracy: {val_results.top5:.4f} ({val_results.top5 * 100:.2f}%)")
print("=" * 60)

### Test on Sample Images

In [5]:
# Test inference on sample images (from test_images or val set)
if TEST_DIR.exists():
    test_samples = list(TEST_DIR.rglob("*.*"))[:10]
    source_label = "test_images"
else:
    test_samples = list((DATASET_DIR / "val").rglob("*.*"))[:10]
    source_label = "val"

print(f"\nTesting inference on {len(test_samples)} sample images ({source_label}):")
print("=" * 60)

for img_path in test_samples:
    results = model.predict(str(img_path), imgsz=IMG_SIZE, verbose=False)
    if results and len(results) > 0:
        result = results[0]
        pred_idx = result.probs.top1
        pred_conf = float(result.probs.top1conf)
        pred_cls = CLASS_NAMES[pred_idx] if pred_idx < len(CLASS_NAMES) else f"class_{pred_idx}"
        print(f"  📷 {img_path.name:<35} → {pred_cls:<16} ({pred_conf:.2%})")

print("\n✓ Inference test completed")
print("=" * 60)

# Save model locally
model_save_path = MODEL_SAVE / "emergency_vehicle_cls_yolov8s.pt"
model.save(str(model_save_path))
print(f"\n✓ Model saved to: {model_save_path}")


Testing inference on 10 sample images (test_images):
  📷 1 (5).jpg                           → emergency        (99.98%)
  📷 2 (6).jpg                           → emergency        (50.08%)
  📷 1 (18).jpg                          → emergency        (99.90%)
  📷 2 (34).jpg                          → non_emergency    (99.40%)
  📷 1 (4).jpg                           → emergency        (99.92%)
  📷 1 (23).jpg                          → emergency        (59.40%)
  📷 1(46).jpg                           → emergency        (100.00%)
  📷 2 (14).jpg                          → non_emergency    (99.99%)
  📷 1 (22).jpg                          → emergency        (99.11%)
  📷 2 (8).jpg                           → non_emergency    (91.23%)

✓ Inference test completed

✓ Model saved to: /content/models/emergency_vehicle_cls_yolov8s.pt


### Save to Google Drive
Mount Drive and copy trained weights so they persist across Colab sessions.
Drive is mounted **only at the end** to avoid I/O latency during training.

In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    print("⚠️ Not running in Google Colab — skipping Drive mount")
    IN_COLAB = False

if IN_COLAB:
    # ── Copy trained weights to Google Drive ──
    DRIVE_OUTPUT = Path("/content/drive/MyDrive/adaptive-traffic-signal/outputs/emergency_vehicle_v1")
    DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

    # Copy best.pt and last.pt from training run
    run_dir = Path("runs/classify/train/weights")  # default YOLO output path
    if not run_dir.exists():
        # search for the actual run directory
        candidates = sorted(Path("runs/classify").glob("*/weights/best.pt"))
        if candidates:
            run_dir = candidates[-1].parent

    if (run_dir / "best.pt").exists():
        shutil.copy2(run_dir / "best.pt", DRIVE_OUTPUT / "best.pt")
        print(f"✓ best.pt  → {DRIVE_OUTPUT / 'best.pt'}")
    if (run_dir / "last.pt").exists():
        shutil.copy2(run_dir / "last.pt", DRIVE_OUTPUT / "last.pt")
        print(f"✓ last.pt  → {DRIVE_OUTPUT / 'last.pt'}")

    # Also copy the local save
    if model_save_path.exists():
        shutil.copy2(model_save_path, DRIVE_OUTPUT / model_save_path.name)
        print(f"✓ {model_save_path.name} → {DRIVE_OUTPUT / model_save_path.name}")

    print(f"\n✅ All weights saved to Google Drive:")
    print(f"   {DRIVE_OUTPUT}")
    for f in sorted(DRIVE_OUTPUT.glob("*.pt")):
        size_mb = f.stat().st_size / 1024**2
        print(f"   • {f.name} ({size_mb:.1f} MB)")
else:
    print("✓ Model saved locally (Drive mount skipped outside Colab)")

Mounted at /content/drive
✓ best.pt  → /content/drive/MyDrive/adaptive-traffic-signal/outputs/emergency_vehicle_v1/best.pt
✓ last.pt  → /content/drive/MyDrive/adaptive-traffic-signal/outputs/emergency_vehicle_v1/last.pt
✓ emergency_vehicle_cls_yolov8s.pt → /content/drive/MyDrive/adaptive-traffic-signal/outputs/emergency_vehicle_v1/emergency_vehicle_cls_yolov8s.pt

✅ All weights saved to Google Drive:
   /content/drive/MyDrive/adaptive-traffic-signal/outputs/emergency_vehicle_v1
   • best.pt (9.8 MB)
   • emergency_vehicle_cls_yolov8s.pt (9.7 MB)
   • last.pt (9.8 MB)


### Summary

✅ **Pipeline Complete:**
1. ✅ Installed ultralytics
2. ✅ Downloaded dataset from GitHub, sorted by CSV labels, 80/20 train/val split
3. ✅ Trained YOLOv8s-cls with `cache=True`, `freeze=9`, AdamW, cosine LR
4. ✅ Evaluated — Top-1/Top-5 accuracy via `model.val()`
5. ✅ Tested inference on sample images
6. ✅ Saved weights to Google Drive (persistent)

**Key Optimizations (matching trafficDetection.ipynb):**
- `yolov8s-cls` — small model, fast fine-tuning (~11M params)
- `cache=True` — dataset loaded to RAM (2-5× training speedup)
- `freeze=9` — backbone frozen (layers 0-8), only Classify head trains
- `workers=4` — saturates CPU data pipelines
- All data on local `/content/` — no Drive I/O during training
- Drive mounted **only at the end** for persistent saves

**Output Locations:**
- Local: `/content/models/emergency_vehicle_cls_yolov8s.pt`
- Drive: `My Drive/adaptive-traffic-signal/outputs/emergency_vehicle_v1/best.pt`